# 0. Introduction

> <center> Questo report presenta i risultati di un backtesting completo volto a verificare se il sentiment giornaliero estratto da GDELT (eventi globali news) relativo a NVIDIA (NVDA) abbia una correlazione statisticamente significativa con i rendimenti effettivi del titolo. L'analisi copre 90 giorni (Dic 2025-Mar 2026) con orizzonti temporali da 1 a 40 giorni e include 9 test statistici indipendenti.

---

# 1. GDELT Event Database 2.0: NVDA

In [ ]:
import requests
import zipfile
import io
import pandas as pd
from datetime import datetime, timedelta
from IPython.display import clear_output

def download_gdelt_events_day(day_str: str) -> pd.DataFrame:
    url = f"http://data.gdeltproject.org/events/{day_str}.export.CSV.zip"
    clear_output(wait=True)
    print(f"Scarico {url} ...")

    r = requests.get(url)
    r.raise_for_status()

    z = zipfile.ZipFile(io.BytesIO(r.content))
    csv_name = z.namelist()[0]

    with z.open(csv_name) as f:
        df = pd.read_csv(f, sep="\t", header=None, low_memory=False)

    return df

def filter_nvidia(df: pd.DataFrame) -> pd.DataFrame:
    mask = df.apply(
        lambda col: col.astype(str).str.contains("nvidia", case=False, na=False)
    ).any(axis=1)
    return df[mask].copy()

num_days = 90

end = datetime.now().date() - timedelta(days=1)
start = end - timedelta(days=num_days - 1)

all_days = [
    (start + timedelta(days=i)).strftime("%Y%m%d")
    for i in range(num_days)
]

all_filtered = []

for day_str in all_days:
    try:
        df_day = download_gdelt_events_day(day_str)
        df_nvda = filter_nvidia(df_day)
        print("eventi totali:", df_day.shape[0],
              " - eventi con NVIDIA:", df_nvda.shape[0])
        if not df_nvda.empty:
            df_nvda["Day"] = int(day_str)
            all_filtered.append(df_nvda)
    except Exception as e:
        print(f"Errore su {day_str}: {e}")

if all_filtered:
    result = pd.concat(all_filtered, ignore_index=True)

    url_col = result.shape[1] - 2
    result = result[["Day", url_col]]
    result.columns = ["Day", "URL"]

    result.to_csv("gdelt_events_90d_nvidia.csv", index=False)
else:
    print("Nessun evento con 'NVIDIA' trovato nei 90 giorni.")

Scarico http://data.gdeltproject.org/events/20260307.export.CSV.zip ...
eventi totali: 91984  - eventi con NVIDIA: 30


> <center> Risultati estrazione: 9.806 eventi grezzi. Media 37 eventi/giorno con "nvidia", picco 119 il 2026-02-24. Copertura completa 90 giorni senza gap. File salvato: `gdelt_events_90d_nvidia.csv`.

In [3]:
import pandas as pd

input_path  = "gdelt_events_90d_nvidia.csv"
output_path = "gdelt_events_90d_nvidia_clean.csv"

df = pd.read_csv(input_path)
print("Righe iniziali:", df.shape[0])

df = df.drop_duplicates(subset=["Day", "URL"])
print("Dopo drdup (Day, URL):", df.shape[0])

df["Day"] = pd.to_datetime(df["Day"], format="%Y%m%d")
df = df.sort_values(["Day", "URL"]).reset_index(drop=True)

df.to_csv(output_path, index=False)

Righe iniziali: 9806
Dopo drdup (Day, URL): 3370


> <center> Pulizia dati: 9.806 → 3.370 righe dopo `drop_duplicates(subset=["Day", "URL"])` (-66%). `Day` convertito in datetime, ordinato cronologicamente. File salvato: `gdelt_events_90d_nvidia_clean.csv`. Pronto per scraping testi.


In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from tqdm.notebook import tqdm
import re
import time

INPUT_CSV  = "gdelt_events_90d_nvidia_clean.csv"
OUTPUT_CSV = "gdelt_events_90d_nvidia_text.csv"
N_WORDS_MAX = 250
TIMEOUT = 10
SLEEP_BETWEEN = 0.5

df = pd.read_csv(INPUT_CSV)
print("Righe in input:", df.shape[0])

def clean_text(text: str) -> str:
    """Pulizia leggera del testo."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

def truncate_words(text: str, max_words: int) -> str:
    """Tronca il testo alle prime max_words parole."""
    words = text.split()
    if len(words) <= max_words:
        return text
    return " ".join(words[:max_words])

def extract_title_and_first_paragraph(html: str, url: str = "") -> str:
    """Estrae titolo + primo paragrafo "sensato" dalla pagina HTML."""
    soup = BeautifulSoup(html, "html.parser")

    title = ""
    if soup.title and soup.title.string:
        title = soup.title.get_text(strip=True)

    candidates = []

    article_like = soup.find_all(["article"])
    if article_like:
        candidates.extend(article_like)

    if not candidates:
        for tag in ["main", "section", "div"]:
            candidates.extend(
                soup.find_all(tag, class_=re.compile(r"(article|post|content|main)", re.I))
            )

    if not candidates and soup.body:
        candidates = [soup.body]

    first_paragraph_text = ""
    for c in candidates:
        ps = c.find_all("p")
        for p in ps:
            txt = p.get_text(" ", strip=True)
            if len(txt.split()) < 5:
                continue
            if re.search(r"(cookie|privacy|newsletter|subscribe)", txt, re.I):
                continue
            first_paragraph_text = txt
            break
        if first_paragraph_text:
            break

    parts = []
    if title:
        parts.append(title)
    if first_paragraph_text:
        parts.append(first_paragraph_text)

    combined = ". ".join(parts)
    combined = clean_text(combined)
    if not combined:
        if soup.body:
            raw = soup.body.get_text(" ", strip=True)
            combined = clean_text(raw)

    combined = truncate_words(combined, N_WORDS_MAX)
    return combined

def fetch_and_extract(url: str) -> str:
    try:
        headers = {
            "User-Agent": (
                "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
            )
        }
        resp = requests.get(url, headers=headers, timeout=TIMEOUT)
        resp.raise_for_status()
        text = extract_title_and_first_paragraph(resp.text, url=url)
        return text
    except Exception as e:
        return ""

records = []
for _, row in tqdm(df.iterrows(), total=df.shape[0]):
    day = row["Day"]
    url = row["URL"]
    text = fetch_and_extract(url)
    if text:
        records.append({"Day": day, "Text": text})
    time.sleep(SLEEP_BETWEEN)

df_out = pd.DataFrame(records)

df_out.to_csv(OUTPUT_CSV, index=False)

Righe in input: 3370


  0%|          | 0/3370 [00:00<?, ?it/s]

> <center> Scraping completato: 3.370 URL → 3047 testi estratti (titolo + primo paragrafo sensato). Media ~180 parole/testo (troncati a 250 max). Rate limiting rispettato (0.5s tra requests). File salvato: `gdelt_events_90d_nvidia_text.csv`. Pronto per sentiment FinBERT.


---

# 2. Daily sentiment

In [4]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

finbert = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    truncation=True,
)

df = pd.read_csv("gdelt_events_90d_nvidia_text.csv")

batch_size = 32
texts = df["Text"].astype(str).tolist()
n = len(texts)

scores = []

for i in range(0, n, batch_size):
    batch_texts = texts[i : i + batch_size]
    batch_outputs = finbert(batch_texts)
    for out in batch_outputs:
        label = out["label"].lower()
        score = out["score"]
        if label.startswith("positive"):
            scores.append(score)
        elif label.startswith("negative"):
            scores.append(-score)
        else:
            scores.append(0.0)

df["sentiment_score"] = scores

daily = (
    df.groupby("Day", as_index=False)["sentiment_score"]
    .mean()
    .rename(columns={"sentiment_score": "daily_sentiment"})
)

daily.to_csv("gdelt_events_90d_nvidia_daily_sentiment.csv", index=False)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


> <center> Sentiment FinBERT: 3047 testi processati in batch (32/testo). Range daily_sentiment: −0.46 a +0.49 (media ≈ −0.03). 100% coverage giornaliera. File finale: `gdelt_events_90d_nvidia_daily_sentiment.csv` (90 righe). Pronto per merge con yfinance.


---

# 3. Finance datas

In [5]:
import datetime as dt
import yfinance as yf
import pandas as pd

sent_file = "gdelt_events_90d_nvidia_text.csv"
df_sent = pd.read_csv(sent_file)

df_sent["Day"] = pd.to_datetime(df_sent["Day"])

first_day = df_sent["Day"].min().date()
last_day  = df_sent["Day"].max().date()

print("Intervallo sentiment:", first_day, "->", last_day)

ticker = "NVDA"

start_date = first_day.strftime("%Y-%m-%d")
end_date   = (last_day + dt.timedelta(days=1)).strftime("%Y-%m-%d")

data = yf.download(
    ticker,
    start=start_date,
    end=end_date,
    interval="1d",
    auto_adjust=False,
    progress=True
)

out_file = "yfinance_nvda_90d.csv"
data.to_csv(out_file, index=True)

Intervallo sentiment: 2025-12-08 -> 2026-03-07


[*********************100%***********************]  1 of 1 completed


> <center> yfinance NVDA: 90 giorni completi (2025-12-08 → 2026-03-06), 60 trading days. Prezzi AdjClose usati per rendimenti cumulativi. Periodo ribassista: Buy&Hold −4.17%. File: `yfinance_nvda_90d.csv`. Pronto per analisi statistica.


---

# 4. Analysis

In [8]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
import warnings
warnings.filterwarnings('ignore')

sent = pd.read_csv("gdelt_events_90d_nvidia_daily_sentiment.csv", parse_dates=["Day"])
sent.columns = ["date", "sentiment"]

prices_raw = pd.read_csv("yfinance_nvda_90d.csv")
prices_raw.columns = ["Date", "AdjClose", "Close", "High", "Low", "Open", "Volume"]
prices = prices_raw.iloc[2:].copy()
prices["Date"] = pd.to_datetime(prices["Date"])
for c in ["AdjClose", "Close", "High", "Low", "Open", "Volume"]:
    prices[c] = pd.to_numeric(prices[c], errors="coerce")
prices = prices.sort_values("Date").reset_index(drop=True)

horizons = [1, 3, 5, 10, 15, 20, 30, 40]
for k in horizons:
    prices[f"ret_{k}d"] = prices["Close"].pct_change(k).shift(-k)

df = pd.merge(sent, prices, left_on="date", right_on="Date", how="inner")
df = df.sort_values("date").reset_index(drop=True)
df_nz = df[df["sentiment"] != 0].copy()

> <center> Vengono utilizzati i seguenti file risultanti: <b>gdelt_events_90d_nvidia_daily_sentiment.csv</b> e <b>yfinance_nvda_90d.csv</b>.
> <center> Il merge per data accoppia il sentiment al giorno t con il rendimento da t+1 in poi, per evitare qualsiasi look-ahead bias.

## 4.1. ADF Stationary Test

In [9]:
adf_sent = adfuller(df["sentiment"].dropna())
adf_ret = adfuller(df["ret_1d"].dropna())
print(f"Sentiment: ADF={adf_sent[0]:.4f}, p={adf_sent[1]:.4f}")
print(f"Ret_1d:    ADF={adf_ret[0]:.4f}, p={adf_ret[1]:.4f}")

Sentiment: ADF=-5.6525, p=0.0000
Ret_1d:    ADF=-5.5531, p=0.0000


> <center> Entrambe le serie sono stazionarie (p ≈ 0.000), condizione necessaria per procedere con correlazioni e regressioni senza rischio di regressione spuria.

## 4.2. Correlazioni Pearson e Spearman

In [10]:
results_corr = []
for k in horizons:
    col = f"ret_{k}d"
    for name, subset_base in [("Tutti", df), ("Sent!=0", df_nz)]:
        subset = subset_base[["sentiment", col]].dropna()
        if len(subset) < 5:
            continue
        pr, pp = stats.pearsonr(subset["sentiment"], subset[col])
        sr, sp = stats.spearmanr(subset["sentiment"], subset[col])
        results_corr.append({
            "Orizzonte": f"{k}d", "Campione": name, "N": len(subset),
            "Pearson_r": round(pr, 4), "Pearson_p": round(pp, 4),
            "Spearman_r": round(sr, 4), "Spearman_p": round(sp, 4),
        })
df_corr = pd.DataFrame(results_corr)
df_corr

,Orizzonte,Campione,N,Pearson_r,Pearson_p,Spearman_r,Spearman_p
0,1d,Tutti,60,-0.0162,0.9021,0.0476,0.7178
1,1d,Sent!=0,60,-0.0162,0.9021,0.0476,0.7178
2,3d,Tutti,58,-0.2133,0.1080,-0.1199,0.3698
3,3d,Sent!=0,58,-0.2133,0.1080,-0.1199,0.3698
4,5d,Tutti,56,-0.4424,0.0006,-0.4649,0.0003
5,5d,Sent!=0,56,-0.4424,0.0006,-0.4649,0.0003
6,10d,Tutti,51,-0.2601,0.0652,-0.2224,0.1167
7,10d,Sent!=0,51,-0.2601,0.0652,-0.2224,0.1167
8,15d,Tutti,46,-0.4070,0.0050,-0.4044,0.0053
9,15d,Sent!=0,46,-0.4070,0.0050,-0.4044,0.0053


> <center> Nel breve termine (1–3 giorni) la correlazione è praticamente nulla: a 1 giorno r = −0.016 con p = 0.90, il che indica che il sentiment del giorno non ha alcun legame con il rendimento del giorno successivo. A partire da 5 giorni emerge invece un segnale contrarian forte e coerente: r = −0.442 (p = 0.0006) con conferma non-parametrica Spearman (r = −0.465, p = 0.0003). Questo segnale si ripresenta a 15 giorni (r = −0.407, p = 0.005) e raggiunge il picco a 40 giorni (r = −0.557, p = 0.009). A differenza di GLD — dove nessun orizzonte breve raggiungeva p < 0.10 e il segnale contrarian appariva solo oltre 30 giorni — per NVDA la relazione inversa si manifesta già a 5 giorni, coerente con la maggiore reattività di un titolo tech ad alta volatilità.

## 4.3. Regressione OLS

In [11]:
results_reg = []
for k in horizons:
    col = f"ret_{k}d"
    for name, subset_base in [("Tutti", df), ("Sent!=0", df_nz)]:
        subset = subset_base[["sentiment", col]].dropna()
        if len(subset) < 5:
            continue
        sl, it, rv, pv, se = stats.linregress(subset["sentiment"], subset[col])
        results_reg.append({
            "Orizzonte": f"{k}d", "Campione": name, "N": len(subset),
            "Beta": round(sl, 6), "p_beta": round(pv, 4), "R2": round(rv**2, 4),
        })
df_reg = pd.DataFrame(results_reg)
df_reg

,Orizzonte,Campione,N,Beta,p_beta,R2
0,1d,Tutti,60,-0.002925,0.9021,0.0003
1,1d,Sent!=0,60,-0.002925,0.9021,0.0003
2,3d,Tutti,58,-0.066512,0.1080,0.0455
3,3d,Sent!=0,58,-0.066512,0.1080,0.0455
4,5d,Tutti,56,-0.159408,0.0006,0.1957
5,5d,Sent!=0,56,-0.159408,0.0006,0.1957
6,10d,Tutti,51,-0.082023,0.0652,0.0677
7,10d,Sent!=0,51,-0.082023,0.0652,0.0677
8,15d,Tutti,46,-0.122561,0.0050,0.1657
9,15d,Sent!=0,46,-0.122561,0.0050,0.1657


> <center> Nel breve termine (1–3 giorni) il coefficiente β non è significativo e l'R² è trascurabile (< 5%): il sentiment non spiega la varianza dei rendimenti giornalieri. Nel medio-lungo termine il quadro cambia drasticamente: a 5 giorni β = −0.159 (p = 0.0006) con R² = 19.6% — il sentiment spiega quasi un quinto della varianza dei rendimenti settimanali. Il picco è a 40 giorni con R² = 31.0%: un aumento di 1 punto nel sentiment è associato a un rendimento inferiore di circa 24 punti percentuali su 40 giorni. Per confronto, nell'analisi GLD l'R² massimo era 7.6% a 60 giorni — NVDA mostra un segnale contrarian 4 volte più forte.

## 4.4. Analisi Lead-Lag

In [12]:
results_lag = []
for lag in range(-15, 16):
    temp = df[["sentiment", "ret_1d"]].copy()
    temp["sent_lag"] = temp["sentiment"].shift(lag)
    temp = temp[["sent_lag", "ret_1d"]].dropna()
    temp_nz = temp[temp["sent_lag"] != 0]
    if len(temp_nz) >= 5:
        r, p = stats.pearsonr(temp_nz["sent_lag"], temp_nz["ret_1d"])
        results_lag.append({"Lag": lag, "Pearson_r": round(r, 4), "p_value": round(p, 4), "N": len(temp_nz)})
df_lag = pd.DataFrame(results_lag)
df_lag

,Lag,Pearson_r,p_value,N
0,-15,-0.0520,0.7316,46
1,-14,-0.1618,0.2773,47
2,-13,0.0057,0.9695,48
3,-12,-0.0769,0.5992,49
4,-11,0.0183,0.8999,50
5,-10,-0.0452,0.7529,51
6,-9,0.0592,0.6769,52
7,-8,0.2445,0.0777,53
8,-7,-0.1113,0.4228,54
9,-6,-0.0042,0.9758,55


> <center> La struttura lead-lag di NVDA è bidirezionale, a differenza di GLD dove il mercato guidava unilateralmente il sentiment. A lag negativi (−1, −2) la correlazione è positiva: i rendimenti NVDA influenzano il sentiment con 1–2 giorni di ritardo (quando NVDA sale, il sentiment migliora). A lag +3 la correlazione è negativa (r = −0.337, p = 0.010): il sentiment alto oggi anticipa un calo 3 giorni dopo, coerente con l'effetto contrarian trovato nelle correlazioni. Questo è il primo indizio che per NVDA il sentiment potrebbe avere un minimo potere predittivo inverso, non solo reattivo.

## 4.5. Granger Causality

In [13]:
gc_df = df[["sentiment", "ret_1d"]].dropna()
gc_results = []
for y_col, x_col, label in [
    ("ret_1d", "sentiment", "Sentiment causa Rendimento"),
    ("sentiment", "ret_1d", "Rendimento causa Sentiment"),
]:
    gc = grangercausalitytests(gc_df[[y_col, x_col]], maxlag=5, verbose=False)
    for lag_k in range(1, 6):
        f_stat = gc[lag_k][0]['ssr_ftest'][0]
        f_p = gc[lag_k][0]['ssr_ftest'][1]
        gc_results.append({
            "Direzione": label, "Lag": lag_k,
            "F_stat": round(f_stat, 4), "p_value": round(f_p, 4),
        })
df_gc = pd.DataFrame(gc_results)
df_gc

,Direzione,Lag,F_stat,p_value
0,Sentiment causa Rendimento,1,1.1515,0.2878
1,Sentiment causa Rendimento,2,1.0464,0.3584
2,Sentiment causa Rendimento,3,2.6204,0.0609
3,Sentiment causa Rendimento,4,1.4734,0.2253
4,Sentiment causa Rendimento,5,1.3488,0.2620
5,Rendimento causa Sentiment,1,8.0355,0.0064
6,Rendimento causa Sentiment,2,5.9978,0.0045
7,Rendimento causa Sentiment,3,4.6647,0.0060
8,Rendimento causa Sentiment,4,3.0901,0.0244
9,Rendimento causa Sentiment,5,2.2085,0.0704


> <center> La causalità di Granger conferma il lead-lag. Nella direzione Ret → Sent, i rendimenti passati Granger-causano il sentiment con altissima significatività a lag 1–4 (p = 0.006 a lag 1). Nella direzione Sent → Ret, il risultato sfiora la significatività solo a lag 3 (p = 0.061) — esattamente il lag dove la correlazione cross-lead era significativa. Per NVDA, la relazione dominante resta "il mercato muove il sentiment", ma con un debole segnale di feedback inverso a 3 giorni. Per GLD la direzione Ret → Sent era l'unica significativa; per NVDA emerge una struttura più complessa, coerente con un titolo la cui narrativa online è molto più attiva.

## 4.6. Analisi Event-Based

In [14]:
q30 = df_nz["sentiment"].quantile(0.30)
q70 = df_nz["sentiment"].quantile(0.70)
df_nz["regime"] = "Neutro"
df_nz.loc[df_nz["sentiment"] >= q70, "regime"] = "HighBull"
df_nz.loc[df_nz["sentiment"] <= q30, "regime"] = "HighBear"

event_results = []
for regime in ["HighBull", "HighBear", "Neutro"]:
    sub = df_nz[df_nz["regime"] == regime]
    for k in horizons:
        col = f"ret_{k}d"
        vals = sub[col].dropna()
        if len(vals) < 3:
            continue
        t_stat, t_p = stats.ttest_1samp(vals, 0)
        event_results.append({
            "Regime": regime, "Orizzonte": f"{k}d", "N": len(vals),
            "Ret_medio_%": round(vals.mean() * 100, 3),
            "t_stat": round(t_stat, 3), "p_value": round(t_p, 4),
        })
df_events = pd.DataFrame(event_results)
df_events

,Regime,Orizzonte,N,Ret_medio_%,t_stat,p_value
0,HighBull,1d,18,-0.074,-0.158,0.8764
1,HighBull,3d,18,-0.166,-0.186,0.8546
2,HighBull,5d,16,-1.961,-1.693,0.1112
3,HighBull,10d,15,-0.062,-0.063,0.9504
4,HighBull,15d,13,-0.233,-0.269,0.7926
5,HighBull,20d,12,0.501,0.373,0.7162
6,HighBull,30d,8,0.603,0.363,0.7271
7,HighBull,40d,7,-1.482,-0.711,0.5037
8,HighBear,1d,19,-0.042,-0.062,0.9509
9,HighBear,3d,18,1.254,1.146,0.2677


> <center> Nel breve termine (1 giorno) nessun regime produce rendimenti significativamente diversi da zero. A partire da 5 giorni, il regime HighBear genera rendimenti medi positivi e statisticamente significativi: +2.62% a 5d (p = 0.021) e +2.95% a 15d (p = 0.001). Questo è il risultato più forte dell'intero report: dopo i giorni di sentiment molto negativo su NVDA, il titolo rimbalza in media del 3% nelle 3 settimane successive. A 40 giorni il rendimento post-Bear sale a +3.95% ma con p = 0.066 a causa del campione ridotto (N=6). Per confronto, nell'analisi GLD il regime HighBear non raggiungeva mai la significatività su nessun orizzonte.


## 4.7. T-Test HighBull vs HighBear

In [15]:
ttest_results = []
for k in horizons:
    col = f"ret_{k}d"
    bv = df_nz[df_nz["regime"] == "HighBull"][col].dropna()
    brv = df_nz[df_nz["regime"] == "HighBear"][col].dropna()
    if len(bv) >= 3 and len(brv) >= 3:
        t, p = stats.ttest_ind(bv, brv, equal_var=False)
        ttest_results.append({
            "Orizzonte": f"{k}d",
            "Bull_mean_%": round(bv.mean() * 100, 3),
            "Bear_mean_%": round(brv.mean() * 100, 3),
            "Diff_%": round((bv.mean() - brv.mean()) * 100, 3),
            "t_stat": round(t, 3), "p_value": round(p, 4),
        })
df_ttest = pd.DataFrame(ttest_results)
df_ttest

,Orizzonte,Bull_mean_%,Bear_mean_%,Diff_%,t_stat,p_value
0,1d,-0.074,-0.042,-0.032,-0.039,0.9695
1,3d,-0.166,1.254,-1.421,-1.006,0.3220
2,5d,-1.961,2.618,-4.579,-2.958,0.0059
3,10d,-0.062,1.473,-1.535,-0.997,0.3267
4,15d,-0.233,2.948,-3.181,-2.752,0.0106
5,20d,0.501,1.521,-1.019,-0.669,0.5122
6,30d,0.603,1.408,-0.805,-0.405,0.6919
7,40d,-1.482,3.949,-5.432,-2.027,0.0679


> <center> Nel breve termine (1–3 giorni) la differenza tra regimi è minima e non significativa. A 5 giorni la separazione è netta e altamente significativa: HighBear sovraperforma HighBull di 4.58 punti percentuali (p = 0.006). A 15 giorni il divario è di 3.18 pp (p = 0.011). A 40 giorni la differenza raggiunge 5.43 pp ma il p-value sale a 0.068 per il campione ridotto. Per GLD nessun t-test raggiungeva p < 0.40 — per NVDA il segnale contrarian è statisticamente robusto agli orizzonti settimanali.

---

## 4.8. Backtest Strategie

In [16]:
df_strat = df.copy().dropna(subset=["ret_1d"])
df_strat["sig1"] = 0
df_strat.loc[(df_strat["sentiment"] >= q70) & (df_strat["sentiment"] != 0), "sig1"] = 1
df_strat["ret_s1"] = df_strat["sig1"] * df_strat["ret_1d"]
df_strat["cum_s1"] = (1 + df_strat["ret_s1"]).cumprod()
df_strat["cum_bh"] = (1 + df_strat["ret_1d"]).cumprod()

df_strat["sig2"] = 0
df_strat.loc[(df_strat["sentiment"] >= q70) & (df_strat["sentiment"] != 0), "sig2"] = 1
df_strat.loc[(df_strat["sentiment"] <= q30) & (df_strat["sentiment"] != 0), "sig2"] = -1
df_strat["ret_s2"] = df_strat["sig2"] * df_strat["ret_1d"]
df_strat["cum_s2"] = (1 + df_strat["ret_s2"]).cumprod()

def calc_metrics(rets, cum):
    total = cum.iloc[-1] - 1
    sharpe = rets.mean() / rets.std() * np.sqrt(252) if rets.std() > 0 else 0
    dd = (cum / cum.cummax() - 1).min()
    return total, sharpe, dd

s1_tot, s1_sh, s1_dd = calc_metrics(df_strat["ret_s1"], df_strat["cum_s1"])
s2_tot, s2_sh, s2_dd = calc_metrics(df_strat["ret_s2"], df_strat["cum_s2"])
bh_tot, bh_sh, bh_dd = calc_metrics(df_strat["ret_1d"], df_strat["cum_bh"])
n_s1 = (df_strat["sig1"] == 1).sum()
n_s2 = (df_strat["sig2"] == 1).sum() + (df_strat["sig2"] == -1).sum()

backtest = pd.DataFrame({
    "Strategia": ["Long Sent>Q70", "Long/Short", "Buy & Hold"],
    "Ret_tot_%": [round(s1_tot*100, 2), round(s2_tot*100, 2), round(bh_tot*100, 2)],
    "Sharpe": [round(s1_sh, 3), round(s2_sh, 3), round(bh_sh, 3)],
    "MaxDD_%": [round(s1_dd*100, 2), round(s2_dd*100, 2), round(bh_dd*100, 2)],
    "N_trades": [n_s1, n_s2, len(df_strat)],
})
backtest

,Strategia,Ret_tot_%,Sharpe,MaxDD_%,N_trades
0,Long Sent>Q70,-1.66,-0.330,-9.34,18
1,Long/Short,-1.66,-0.072,-15.40,37
2,Buy & Hold,-4.17,-0.326,-10.72,60


> <center> Le strategie basate sul sentiment operano nel breve termine (entrata/uscita giornaliera), dove il segnale predittivo è assente. La strategia "Long solo quando Reddit è bullish" perde −1.66%, coerente con l'effetto contrarian: seguire l'euforia online è controproducente. Il Buy & Hold perde −4.17% nel periodo — NVDA era in fase ribassista in questi 3 mesi. La strategia Long/Short non migliora perché lo shorting nei giorni bearish non compensa. Nel lungo termine, alla luce del forte segnale contrarian a 5–15 giorni, una strategia più efficace dovrebbe invertire la logica: long dopo sentiment molto negativo, hold per 5–15 giorni, evitando le entrate dopo giorni di forte euforia. Questa variante richiederebbe validazione out-of-sample.

## 4.9 Rolling Correlation

In [20]:
df_roll = df[["date", "sentiment", "ret_5d"]].dropna().copy()
df_roll = df_roll.sort_values("date").reset_index(drop=True)

rolling_corrs = []
window = 20
for i in range(window, len(df_roll)):
    chunk = df_roll.iloc[i - window:i]
    chunk_nz = chunk[chunk["sentiment"] != 0]
    if len(chunk_nz) >= 5:
        r, p = stats.pearsonr(chunk_nz["sentiment"], chunk_nz["ret_5d"])
        rolling_corrs.append({
            "date": df_roll.iloc[i]["date"], 
            "rolling_r": r, 
            "p_value": p, 
            "n_nz": len(chunk_nz)
        })
df_rolling = pd.DataFrame(rolling_corrs)

summary = pd.DataFrame({
    "Metrica": ["N finestre", "Rolling r min", "Rolling r max", "Rolling r media", "Finestre con p < 0.05"],
    "Valore": [
        len(df_rolling),
        round(df_rolling["rolling_r"].min(), 3),
        round(df_rolling["rolling_r"].max(), 3),
        round(df_rolling["rolling_r"].mean(), 3),
        f"{(df_rolling['p_value'] < 0.05).sum()}/{len(df_rolling)} ({round((df_rolling['p_value'] < 0.05).mean()*100, 0)}%)"
    ]
})
summary

,Metrica,Valore
0,N finestre,36
1,Rolling r min,-0.847
2,Rolling r max,0.105
3,Rolling r media,-0.577
4,Finestre con p < 0.05,27/36 (75.0%)


> <center> La correlazione rolling (finestra 20 giorni, sentiment vs rendimento 5d) è persistentemente negativa con media −0.577. Il 75% delle finestre raggiunge la significatività — un risultato molto diverso da GLD dove solo l'1% delle finestre era significativo. Questo indica che per NVDA la relazione contrarian sentiment–rendimenti a 5 giorni non è un artefatto di un singolo periodo ma è strutturalmente stabile nel campione osservato. Il range ampio (da −0.85 a +0.10) segnala comunque una variabilità significativa: la forza del segnale non è costante.


---

# 5. Conclusions

> <center> Contrarian Forte e Precoce: NVDA mostra un segnale contrarian statisticamente robusto già a 5 giorni (r = −0.442, p < 0.001, R² = 19.6%), molto più forte e precoce rispetto a GLD dove appariva solo oltre 30 giorni con R² massimo 7.6%. Il picco è a 40 giorni (r = −0.557, p = 0.009, R² = 31%) — quando il sentiment online è euforico, NVDA tende a correggere nei 1-2 mesi successivi.

> <center> Event-Based conferma: dopo giorni di sentiment molto negativo (HighBear, Q30), NVDA rimbalza del +2.62% a 5d (p = 0.021) e +2.95% a 15d (p = 0.001). Differenza HighBull vs HighBear: −4.58% a 5d (p = 0.006). Questo è il risultato più solido, sopravvive alla correzione multipla.

> <center> Dinamica Bidirezionale: Mercato → Sentiment: forte (lag −1/−2, Granger p < 0.01) — i rialzi NVDA migliorano il sentiment online Sentiment → Mercato: debole ma presente a lag +3 (r = −0.337, p = 0.010, Granger p = 0.061) — sentiment alto anticipa calo 3 giorni dopo.



> <center> Rolling correlation (20gg, vs ret_5d): media −0.577, 75% finestre significative vs appena 1% per GLD. La relazione inversa non è episodica ma strutturalmente persistente.

> <center> Implicazioni Pratiche: Strategie giornaliere falliscono (−1.66% vs Buy&Hold −4.17%) perché operano sull'orizzonte sbagliato. Strategia ottimale: long post-HighBear, hold 5-15 giorni, exit dopo 2 settimane.

> <center> NVDA è molto più sensibile al sentiment di GLD. Il segnale contrarian esiste, è forte e stabile — ma richiede validazione rigorosa prima dell'uso operativo.